# Atelier 3.B — L'agent du service après-vente ★

**Durée : 75 min** · Projet fil rouge, incrément du Jour 3

Le tri (Jour 1) et l'assistant documentaire (Jour 2) savent **répondre** ; l'agent sait **agir**. Il traite une demande de remboursement de bout en bout :

1. il consulte le CRM de Fictiva, exposé par un **serveur MCP** ;
2. il calcule le montant avec un outil déterministe qui applique la politique de retours ;
3. il crée un ticket, après **validation humaine** ;
4. il rédige un brouillon de réponse au client.

L'agent est un graphe **LangGraph** ; les briques sont dans `agents_fictiva.py`, `crm.py` et `serveur_crm_mcp.py`.

In [ ]:
import json
import sys

import pandas as pd
from langchain_core.messages import HumanMessage
from langgraph.types import Command

import agents_fictiva as af
import crm
import outils_llm as llm

MODELE = llm.choisir_modele()   # le modèle doit savoir appeler des outils
print("Modèle :", MODELE, "| date du scénario :", crm.DATE_REFERENCE)

def formater(demande):
    return f"De : {demande['expediteur']} ({demande['entreprise']})\nObjet : {demande['objet']}\n\n{demande['corps']}"

## 1. Le CRM simulé

Clients, commandes et catalogue de Fictiva, cohérents avec les emails du Jour 1 et les documents du Jour 2. La date du scénario est fixée au 23 octobre 2026 pour que les délais de retour restent reproductibles.

In [ ]:
pd.DataFrame([{"numero_client": n, **c} for n, c in crm.DONNEES["clients"].items()])

In [ ]:
pd.DataFrame([{"commande": n, "client": c["client"], "statut": c["statut"], "livraison": c["date_livraison"],
               "articles": ", ".join(f"{l['quantite']} × {l['reference']}" for l in c["lignes"])}
              for n, c in crm.DONNEES["commandes"].items()])

## 2. Le serveur MCP

Le **Model Context Protocol** standardise la façon dont un agent découvre et appelle des outils. Le serveur du CRM n'expose que trois outils **en lecture** : c'est le principe du moindre privilège.

In [ ]:
print(open("serveur_crm_mcp.py", encoding="utf-8").read())

N'importe quel client MCP peut utiliser ce serveur : un agent, un IDE, un assistant de bureau. Voici le dialogue du protocole, sans framework : on lance le serveur, on liste ses outils, on en appelle un.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

parametres = StdioServerParameters(command=sys.executable, args=["serveur_crm_mcp.py"])
async with stdio_client(parametres) as (lecture, ecriture):
    async with ClientSession(lecture, ecriture) as session:
        await session.initialize()
        for outil in (await session.list_tools()).tools:
            print(f"{outil.name} {list(outil.inputSchema['properties'])}")
        resultat = await session.call_tool("consulter_commande", {"numero_commande": "CMD-2026-0415"})
        print("\n", resultat.content[0].text[:500])

## 3. Les outils locaux

- `calculer_remboursement` applique la politique de retours **en Python**. Le calcul d'un montant ne se confie pas au modèle : un outil déterministe est exact, testable et auditable.
- `creer_ticket` écrit dans le CRM. C'est une action engageante : l'outil **suspend** l'agent (`interrupt`) jusqu'à la décision d'un opérateur.

In [ ]:
print(crm.calculer_remboursement("CMD-2026-0415", ["KB-80"], defectueux=False))
print(crm.calculer_remboursement("CMD-2026-0402", ["SE-700"], defectueux=False))
print(crm.calculer_remboursement("CMD-2026-0409", ["CE-20"], defectueux=False, consommables_ouverts=True))

## 4. Le graphe de l'agent

Deux nœuds : `agent` (le modèle, avec la liste des outils) et `outils` (l'exécution). Tant que le modèle demande des outils, on boucle ; dès qu'il répond sans outil, on termine. La mémoire (*checkpointer*) conserve l'état de chaque conversation : c'est elle qui permet la pause et la reprise.

In [ ]:
outils_mcp = await af.outils_crm_mcp()
OUTILS = outils_mcp + [af.calculer_remboursement, af.creer_ticket]
print([o.name for o in OUTILS])

agent = af.construire_agent(OUTILS, modele=MODELE)
print(agent.get_graph().draw_mermaid())

Le prompt système donne la **méthode**, pas les réponses :

In [ ]:
print(af.PROMPT_SAV)

## 5. Traiter une demande, avec validation humaine

In [ ]:
demandes = af.charger_demandes()
demande = demandes[0]
print(formater(demande))

configuration = af.config(demande["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)

L'agent s'est arrêté avant de créer le ticket. L'opérateur voit exactement ce qui sera écrit :

In [ ]:
for interruption in resultat.get("__interrupt__", []):
    print(json.dumps(interruption.value, ensure_ascii=False, indent=2))

In [ ]:
resultat = await agent.ainvoke(Command(resume="oui"), configuration)   # l'opérateur valide
af.afficher_trajectoire(resultat["messages"])
print("Tickets :", crm.TICKETS)

## 6. Quand l'opérateur refuse

La commande de Golay Architectes n'a jamais été livrée : ce n'est pas un retour. Voyons ce que l'agent propose, puis refusons le ticket avec une consigne.

In [ ]:
demande = demandes[5]
configuration = af.config(demande["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)
for interruption in resultat.get("__interrupt__", []):
    print(json.dumps(interruption.value, ensure_ascii=False, indent=2))

In [ ]:
consigne = ("non : la politique de retours ne s'applique pas à une commande non livrée et aucun remboursement "
            "ne doit être promis. Crée un ticket « reclamation » sans montant, à transmettre au responsable du service client.")
resultat = await agent.ainvoke(Command(resume=consigne), configuration)
while resultat.get("__interrupt__"):
    print("Nouvelle proposition :", json.dumps(resultat["__interrupt__"][0].value, ensure_ascii=False))
    resultat = await agent.ainvoke(Command(resume="oui"), configuration)
print("\n" + resultat["messages"][-1].content)

**À retenir.** La validation humaine n'est pas une formalité : sur les décisions engageantes, elle rattrape les erreurs de l'agent. Ici, l'opérateur a empêché une promesse de remboursement que la politique ne prévoit pas.

Relisez pourtant le brouillon : l'agent peut encore affirmer des règles qui n'existent nulle part. Il connaît le CRM, mais **pas la documentation** de Fictiva.

## 7. Toutes les demandes

On traite les autres demandes en validant automatiquement les tickets, puis on compare les montants calculés aux montants attendus.

In [ ]:
async def traiter(demande, decision="oui"):
    configuration = af.config(demande["id"] + "-lot")
    resultat = await agent.ainvoke({"messages": [HumanMessage(formater(demande))]}, configuration)
    while resultat.get("__interrupt__"):
        resultat = await agent.ainvoke(Command(resume=decision), configuration)
    return resultat

lignes = []
for demande in demandes[1:5]:
    resultat = await traiter(demande)
    calculs = [json.loads(m.content) for m in resultat["messages"] if getattr(m, "name", "") == "calculer_remboursement"]
    dernier = calculs[-1] if calculs else {}
    lignes.append({"demande": demande["id"], "attendu": demande["attendu"]["montant_chf"],
                   "calcule": dernier.get("montant_chf"), "validation_requise": dernier.get("validation_requise"),
                   "appels_d_outils": sum(len(getattr(m, "tool_calls", [])) for m in resultat["messages"])})
pd.DataFrame(lignes)

## 8. La mémoire de conversation

Le fil de conversation de la première demande est conservé : on peut poursuivre l'échange sans tout réexpliquer.

In [ ]:
configuration = af.config(demandes[0]["id"])
resultat = await agent.ainvoke({"messages": [HumanMessage(
    "La cliente demande si elle peut recevoir un avoir plutôt qu'un virement. Réponds-lui en deux phrases, sans créer de nouveau ticket.")]},
    configuration)
while resultat.get("__interrupt__"):   # si l'agent tente malgré tout de créer un ticket, l'opérateur refuse
    resultat = await agent.ainvoke(Command(resume="non : aucun nouveau ticket n'est nécessaire."), configuration)
print(resultat["messages"][-1].content)

**À vérifier.** L'article 6 de la politique de retours prévoit un remboursement « par virement ou sous forme d'avoir ». La réponse de l'agent est-elle conforme ? Sans accès à la documentation, l'agent répond de mémoire : s'il a juste, c'est par chance. Au Jour 4, l'assistant documentaire du Jour 2 devient l'un de ses outils.

✏️ Écrivez une demande ambiguë (un client qui ne donne ni numéro de client ni numéro de commande, et cite un produit commandé plusieurs fois). L'agent pose-t-il les bons appels ? Que manque-t-il au prompt ?

## À retenir

- Le modèle **orchestre**, les outils **exécutent** : les montants viennent d'un calcul déterministe, pas du modèle.
- MCP sépare l'agent de ses sources : le même serveur CRM peut servir d'autres agents et d'autres applications.
- Moindre privilège : lecture seule via MCP, écriture réservée à un outil soumis à **validation humaine**.
- La mémoire par fil de conversation permet la pause, la reprise et le suivi d'un dossier.